In [129]:
import numpy as np
import pandas as pd

In [130]:
import joblib

In [131]:
import warnings
warnings.filterwarnings('ignore')

In [132]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

In [133]:
CAREERS = {
    "Software Engineer": {
        "description": "Design, build, and maintain software systems & applications.",
        "key_subjects": ["Mathematics", "Computer Science", "Physics"],
        "key_interests": ["technology", "gaming", "problem_solving", "coding"],
        "key_personality": ["investigative", "realistic", "conventional"],
        "required_skills": ["programming", "logical_thinking", "mathematics"],
        "salary_range": "₹6–40 LPA",
        "growth_outlook": "Very High",
    },
    "Data Scientist": {
        "description": "Extract insights from complex data using statistics and ML.",
        "key_subjects": ["Mathematics", "Statistics", "Computer Science"],
        "key_interests": ["research", "mathematics", "technology", "problem_solving"],
        "key_personality": ["investigative", "artistic", "conventional"],
        "required_skills": ["statistics", "programming", "critical_thinking"],
        "salary_range": "₹8–45 LPA",
        "growth_outlook": "Very High",
    },
    "Doctor / Medical Professional": {
        "description": "Diagnose, treat, and prevent illness; serve patient health.",
        "key_subjects": ["Biology", "Chemistry", "Physics"],
        "key_interests": ["health", "science", "helping_others", "research"],
        "key_personality": ["social", "investigative", "realistic"],
        "required_skills": ["empathy", "biology", "critical_thinking"],
        "salary_range": "₹8–50 LPA",
        "growth_outlook": "High",
    },
    "Civil Engineer": {
        "description": "Plan and construct infrastructure — bridges, roads, buildings.",
        "key_subjects": ["Mathematics", "Physics", "Chemistry"],
        "key_interests": ["design", "construction", "problem_solving", "outdoors"],
        "key_personality": ["realistic", "conventional", "investigative"],
        "required_skills": ["mathematics", "design", "project_management"],
        "salary_range": "₹4–20 LPA",
        "growth_outlook": "Moderate",
    },
    "Graphic Designer / UX Designer": {
        "description": "Create visual concepts and intuitive digital experiences.",
        "key_subjects": ["Fine Arts", "Computer Science", "English"],
        "key_interests": ["art", "design", "technology", "creativity"],
        "key_personality": ["artistic", "social", "enterprising"],
        "required_skills": ["creativity", "design_tools", "communication"],
        "salary_range": "₹3–25 LPA",
        "growth_outlook": "High",
    },
    "Business Analyst / MBA": {
        "description": "Bridge business needs with technology and data-driven strategy.",
        "key_subjects": ["Economics", "Mathematics", "Commerce"],
        "key_interests": ["business", "finance", "problem_solving", "leadership"],
        "key_personality": ["enterprising", "conventional", "social"],
        "required_skills": ["analytical_thinking", "communication", "leadership"],
        "salary_range": "₹6–35 LPA",
        "growth_outlook": "High",
    },
    "Lawyer / Legal Professional": {
        "description": "Advise clients, represent them, and navigate legal systems.",
        "key_subjects": ["English", "History", "Political Science"],
        "key_interests": ["debate", "reading", "justice", "politics"],
        "key_personality": ["enterprising", "social", "conventional"],
        "required_skills": ["communication", "critical_thinking", "research"],
        "salary_range": "₹4–30 LPA",
        "growth_outlook": "Moderate",
    },
    "Psychologist / Counselor": {
        "description": "Study human behaviour and support mental health & well-being.",
        "key_subjects": ["Biology", "Sociology", "English"],
        "key_interests": ["helping_others", "research", "human_behaviour", "health"],
        "key_personality": ["social", "investigative", "artistic"],
        "required_skills": ["empathy", "communication", "research"],
        "salary_range": "₹3–15 LPA",
        "growth_outlook": "High",
    },
    "Teacher / Educator": {
        "description": "Inspire and educate students across subjects and levels.",
        "key_subjects": ["English", "Mathematics", "Science"],
        "key_interests": ["helping_others", "leadership", "public_speaking", "research"],
        "key_personality": ["social", "artistic", "conventional"],
        "required_skills": ["communication", "patience", "leadership"],
        "salary_range": "₹3–12 LPA",
        "growth_outlook": "Moderate",
    },
    "Entrepreneur / Business Owner": {
        "description": "Identify opportunities and build businesses from scratch.",
        "key_subjects": ["Economics", "Commerce", "Mathematics"],
        "key_interests": ["business", "innovation", "technology", "leadership"],
        "key_personality": ["enterprising", "artistic", "social"],
        "required_skills": ["leadership", "risk_taking", "communication"],
        "salary_range": "Variable (₹2L–∞)",
        "growth_outlook": "Depends on venture",
    },
    "Mechanical Engineer": {
        "description": "Design and develop mechanical systems and devices.",
        "key_subjects": ["Mathematics", "Physics", "Chemistry"],
        "key_interests": ["machines", "design", "problem_solving", "outdoors"],
        "key_personality": ["realistic", "investigative", "conventional"],
        "required_skills": ["mathematics", "design", "logical_thinking"],
        "salary_range": "₹4–22 LPA",
        "growth_outlook": "Moderate",
    },
    "Journalist / Content Creator": {
        "description": "Research, write, and share stories across media channels.",
        "key_subjects": ["English", "History", "Political Science"],
        "key_interests": ["writing", "travel", "politics", "creativity"],
        "key_personality": ["artistic", "enterprising", "social"],
        "required_skills": ["communication", "research", "creativity"],
        "salary_range": "₹2–20 LPA",
        "growth_outlook": "Moderate",
    }

}


In [134]:
RIASEC_TYPES = ["realistic", "investigative", "artistic", "social", "enterprising", "conventional"]

In [135]:
ALL_INTERESTS = [
    "technology", "art", "music", "sports", "science", "mathematics", "writing",
    "coding", "gaming", "research", "design", "business", "finance", "health",
    "helping_others", "politics", "travel", "outdoors", "debate", "leadership",
    "public_speaking", "human_behaviour", "problem_solving", "creativity",
    "machines", "construction", "innovation", "reading", "justice","people interaction","service",
    "culture","investing","analytics","banking","numbers","social impact","law", "goverance",
    "business", "entrepreneurship", "markets", "sales"
]

In [136]:
ALL_SKILLS = [
    "programming", "mathematics", "communication", "leadership", "research",
    "design_tools", "statistics", "critical_thinking", "empathy", "biology",
    "logical_thinking", "analytical_thinking", "creativity", "project_management",
    "patience", "risk_taking","numerical_ability", "data_analysis", "risk_assessment",
    "leadership", "decision-making", "communication", "customer_service", "problem-solving"
]

In [137]:
def generate_synthetic_dataset(n_samples: int = 3000, random_state: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    records = []

    career_list = list(CAREERS.keys())

    for _ in range(n_samples):
        target_career = rng.choice(career_list)
        info = CAREERS[target_career]
        base_score = rng.integers(50, 95)
        scores = {
            "score_mathematics":      int(np.clip(base_score + rng.integers(-20, 20), 40, 100)),
            "score_science":          int(np.clip(base_score + rng.integers(-15, 15), 40, 100)),
            "score_computer_science": int(np.clip(base_score + rng.integers(-20, 20), 40, 100)),
            "score_english":          int(np.clip(base_score + rng.integers(-15, 15), 40, 100)),
            "score_social_studies":   int(np.clip(base_score + rng.integers(-20, 20), 40, 100)),
            "score_biology":          int(np.clip(base_score + rng.integers(-15, 15), 40, 100)),
            "score_commerce":         int(np.clip(base_score + rng.integers(-20, 20), 40, 100)),
        }

        for subj in info["key_subjects"]:
            col = "score_" + subj.lower().replace(" ", "_")
            if col in scores:
                scores[col] = int(np.clip(scores[col] + rng.integers(5, 20), 30, 100))

                
        personality = {f"personality_{t}": int(rng.integers(1, 8)) for t in RIASEC_TYPES}
        for pt in info["key_personality"]:
            personality[f"personality_{pt}"] = int(np.clip(personality[f"personality_{pt}"] + rng.integers(2, 4), 1, 10))

        
        interests = {f"interest_{i}": 0 for i in ALL_INTERESTS}
       
        for ki in info["key_interests"]:
            if f"interest_{ki}" in interests:
                interests[f"interest_{ki}"] = 1
        
        for i in ALL_INTERESTS:
            if rng.random() < 0.25:
                interests[f"interest_{i}"] = 1

    
        skills = {f"skill_{s}": 0 for s in ALL_SKILLS}
        for rs in info["required_skills"]:
            if f"skill_{rs}" in skills:
                skills[f"skill_{rs}"] = 1
        for s in ALL_SKILLS:
            if rng.random() < 0.2:
                skills[f"skill_{s}"] = 1

       
        meta = {
            "gpa_overall": round(
                (scores["score_mathematics"] + scores["score_science"] +
                 scores["score_english"]) / 3 / 10, 2
            ),
            "extracurricular_count": int(rng.integers(0, 7)),
            "prefers_teamwork":      int(rng.random() < 0.5),
            "prefers_remote":        int(rng.random() < 0.4),
            "risk_tolerance":        int(rng.integers(1, 10)),
            "career_label":          target_career,
        }

        record = {**scores, **personality, **interests, **skills, **meta}
        records.append(record)

    df = pd.DataFrame(records)
    print(f"Generated dataset: {df.shape[0]} samples × {df.shape[1]} features")
    #print(df)
    return df
df = generate_synthetic_dataset(n_samples=4000)

Generated dataset: 4000 samples × 83 features


In [138]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    
    df1 = df.copy()

    stem_cols = ["score_mathematics", "score_science",
                 "score_computer_science", "score_biology"]
    df1["stem_affinity"] = df1[[c for c in stem_cols if c in df1.columns]].mean(axis=1)

    arts_cols = ["score_english", "score_social_studies", "score_commerce"]
    df1["arts_affinity"] = df1[[c for c in arts_cols if c in df1.columns]].mean(axis=1)

    interest_cols = [c for c in df1.columns if c.startswith("interest_")]
    df1["interest_diversity"] = df1[interest_cols].sum(axis=1)

    skill_cols = [c for c in df1.columns if c.startswith("skill_")]
    df1["skill_breadth"] = df1[skill_cols].sum(axis=1)

    df1["people_orientation"] = (
        df1.get("personality_social", 0) +
        df1.get("personality_enterprising", 0) +
        df1.get("interest_helping_others", 0)
    )
    df1["things_orientation"] = (
        df1.get("personality_realistic", 0) +
        df1.get("personality_investigative", 0) +
        df1.get("interest_machines", 0)
    )
    #print(df1)
    return df1


In [139]:
df2 = engineer_features(df)

In [140]:
def preprocess(df: pd.DataFrame):
    df = engineer_features(df)
    label_col = "career_label"
    X = df.drop(columns=[label_col])
    y = df[label_col]

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    #print(X_scaled)
    #print(y_enc)
    #print(le.classes_)
    #print(scaler)

    return X_scaled, y_enc, le, scaler, X.columns.tolist()

In [141]:
#preprocess(df2)

In [142]:
def build_model_ensemble(X_scaled, y_enc, le, scaler, feature_names):
    rf=RandomForestClassifier(n_estimators=300,  max_depth=None, min_samples_split=4,min_samples_leaf=2,
        max_features="sqrt",class_weight="balanced",random_state=42,n_jobs=-1,
    )
    gb=GradientBoostingClassifier(n_estimators=300,learning_rate=0.08,max_depth=5, subsample= 0.8, random_state=42) 
    ensemble = VotingClassifier(estimators=[('rf', rf), ('gb', gb)], voting='soft', weights=[2, 1])
    #print(ensemble)
    return ensemble

In [143]:
X_scaled, y_enc, le, scaler, feature_names = preprocess(df2) 

ensemble = build_model_ensemble(X_scaled, y_enc, le, scaler, feature_names)

In [144]:
def train_model(X_train,y_train):
    model=build_model_ensemble(X_scaled, y_enc, le, scaler, feature_names)
    model.fit(X_train,y_train)
    return model

In [145]:
def evaluate_model(model,X_train,y_train,le):
    y_pred=model.predict(X_train)
    model_accuracy = accuracy_score(y_train, y_pred)
    print("-" * 50)
    print(f"Model Accuracy: {model_accuracy*100:.2f}")
    print("-" * 50)
    print("CLasifictation Report")
    print(classification_report(y_train, y_pred, target_names=le.classes_))
    print("-" * 50)
    return model_accuracy

In [146]:
#X_scaled, y_enc, le, scaler, feature_names = preprocess(df2)
#model=train_model(X_scaled,y_enc)
#evaluate_model(model,X_scaled,y_enc,le)

In [147]:
def rule_based_score(student_profile: dict, career: str) -> float:
    info = CAREERS[career]
    score = 0.0
    checks = 0

    student_interests = student_profile.get("interests", [])
    interest_match = sum(1 for ki in info["key_interests"] if ki in student_interests)
    score += interest_match / max(len(info["key_interests"]), 1)
    checks += 1

    student_skills = student_profile.get("skills", [])
    skill_match = sum(1 for rs in info["required_skills"] if rs in student_skills)
    score += skill_match / max(len(info["required_skills"]), 1)
    checks += 1

  
    student_personality = student_profile.get("personality", {})
    personality_match = sum(
        student_personality.get(pt, 0) / 10
        for pt in info["key_personality"]
    )
    score += personality_match / max(len(info["key_personality"]), 1)
    checks += 1

    subject_scores = student_profile.get("subject_scores", {})
    above_threshold = sum(
        1 for subj in info["key_subjects"]
        if subject_scores.get(subj, 0) >= 60
    )
    score += above_threshold / max(len(info["key_subjects"]), 1)
    checks += 1

    return round(score / checks, 4)

In [148]:
def profile_to_feature_vector(student_profile: dict, feature_names: list) -> np.ndarray:
        
    subj = student_profile.get("subject_scores", {})
    pers = student_profile.get("personality", {})
    interests_list = student_profile.get("interests", [])
    skills_list = student_profile.get("skills", [])
    meta = student_profile.get("meta", {})

    row = {}

    for subj_name, default in [
        ("Mathematics", 60), ("Science", 60), ("Computer Science", 50),
        ("English", 60), ("Social Studies", 55), ("Biology", 55), ("Commerce", 50)
    ]:
        col = "score_" + subj_name.lower().replace(" ", "_")
        row[col] = subj.get(subj_name, default)

   
    for t in RIASEC_TYPES:
        row[f"personality_{t}"] = pers.get(t, 5)


    for i in ALL_INTERESTS:
        row[f"interest_{i}"] = 1 if i in interests_list else 0

   
    for s in ALL_SKILLS:
        row[f"skill_{s}"] = 1 if s in skills_list else 0

   
    row["gpa_overall"] = meta.get("gpa_overall", 6.0)
    row["extracurricular_count"] = meta.get("extracurricular_count", 2)
    row["prefers_teamwork"] = int(meta.get("prefers_teamwork", True))
    row["prefers_remote"] = int(meta.get("prefers_remote", False))
    row["risk_tolerance"] = meta.get("risk_tolerance", 5)

    df_row = pd.DataFrame([row])
    df_row = engineer_features(df_row)

    for col in feature_names:
        if col not in df_row.columns:
            df_row[col] = 0
    df_row = df_row[feature_names]

    return df_row.values


In [149]:
def recommend_careers(student_profile: dict, model, le: LabelEncoder, scaler: StandardScaler, feature_names: list,
    top_n: int = 5, ml_weight: float = 0.65, rule_weight: float = 0.35,) -> list[dict]:
   
    fv = profile_to_feature_vector(student_profile, feature_names)
    fv_scaled = scaler.transform(fv)
    probs = model.predict_proba(fv_scaled)[0]         
    career_names = le.inverse_transform(np.arange(len(probs)))

    results = []
    student_skills = set(student_profile.get("skills", []))

    for career, ml_prob in zip(career_names, probs):
        rule_score = rule_based_score(student_profile, career)
        final_score = ml_weight * ml_prob + rule_weight * rule_score

        required = set(CAREERS[career]["required_skills"])
        skill_gap = list(required - student_skills)

        results.append({
            "career":          career,
            "ml_prob":         round(float(ml_prob), 4),
            "rule_score":      round(rule_score, 4),
            "final_score":     round(final_score, 4),
            "confidence_pct":  round(final_score * 100, 1),
            "description":     CAREERS[career]["description"],
            "salary_range":    CAREERS[career]["salary_range"],
            "growth_outlook":  CAREERS[career]["growth_outlook"],
            "skill_gap":       skill_gap,
        })

    results.sort(key=lambda x: x["final_score"], reverse=True)
    return results[:top_n]


In [150]:
def save_model(model, le, scaler, feature_names, path="career_model.pkl"):
    payload = {
        "model": model,
        "label_encoder": le,
        "scaler": scaler,
        "feature_names": feature_names,
    }
    joblib.dump(payload, path)
    print(f"Model saved as : {path}")


In [151]:
def load_model(path="career_model.pkl"):
    payload = joblib.load(path)
    print(f"Model loaded from {path}")
    return payload["model"], payload["label_encoder"], payload["scaler"], payload["feature_names"]

In [152]:
def print_result(recommendations: list[dict], student_name: str = "Student"):
    print("-" * 50)
    print(f"Career Result for: {student_name}")
    print("-" * 50)

    for rank, rec in enumerate(recommendations, 1):
        gap_str = ", ".join(rec["skill_gap"]) if rec["skill_gap"] else "None you're ready!"
        print(f"#{rank}  {rec['career']}")
        print(f"Match Score  : {rec['confidence_pct']}%")
        print(f"Description  : {rec['description']}")
        print(f"Salary Range : {rec['salary_range']}")
        print(f"Growth       : {rec['growth_outlook']}")
        print(f"Skill Gap    : {gap_str}")
        print("-" * 50)

    print("Focus on closing skill gaps for your #1 match!")
    print()

In [ ]:
def run_pipline():
    print("-"*50)
    print("AI - Driven Career Recommendations System")
    print("-"*50)

    df = generate_synthetic_dataset(n_samples=4000)

    X_scaled,y_enc, le, scaler, feature_names = preprocess(df)
    ##print(f"Features: {feature_names}")

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
    ##print(f"Train set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

    print("Training the model ensemble")
    model=train_model(X_train.values,y_train)

    print("Cross-validation")
    rf_quick = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    cv_scores = cross_val_score(rf_quick, X_train, y_train, cv=5, scoring="accuracy")
    print(f"CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

    print("Evaluating on training data")
    evaluate_model(model,X_train,y_train,le)

    print("Saving the model")
    save_model(model, le, scaler, feature_names)

    sample_profile_1= {
        "name": "Rumaisa",
        "subject_scores": {
            "Mathematics": 90,
            "Science": 80,
            "Computer Science": 90,
            "English": 70,
            "Social Studies": 65,
            "Biology": 75,
            "Commerce": 60
        },
        "personality": {
            "realistic": 7,
            "investigative": 8,
            "artistic": 5,
            "social": 6,
            "enterprising": 7,
            "conventional": 6
        },
        "interests": ["technology", "mathematics", "problem_solving", "coding"],
        "skills": ["programming", "logical_thinking", "mathematics"],
        "meta": {
            "gpa_overall": 8.6,
            "extracurricular_count": 3,
            "prefers_teamwork": True,
            "prefers_remote": False,
            "risk_tolerance": 6
        }
    }

    sample_profile_2 = {
        "name": "Arshad",
        "subject_scores": {
            "Mathematics": 85,
            "Science": 75,
            "Computer Science": 96,
            "English": 70,
            "Social Studies": 65,
            "Biology": 55,
            "Commerce": 60
        },
        "personality": {
            "realistic": 8,
            "investigative": 8,
            "artistic": 9,
            "social": 3,
            "enterprising": 5,
            "conventional": 8
        },
        "interests": ["business", "finance", "problem_solving", "leadership"],
        "skills": ["numerical_ability", "risk_assessment", "analytical_thinking", "communication", "leadership"],
        "meta": {
            "gpa_overall": 8.2,
            "extracurricular_count": 4,
            "prefers_teamwork": True,
            "prefers_remote": False,
            "risk_tolerance": 5
        }
    }
    print()
    print()
    print("-"*50)
    print("Generating career recommendations for sample profile 1")
    recommendations = recommend_careers(sample_profile_1, model, le, scaler, feature_names)
    print_result(recommendations, student_name=sample_profile_1["name"])
    print("-"*50)
    print()
    print()
    print("-"*50)
    print("Generating career recommendations for sample profile 2")
    recommendations = recommend_careers(sample_profile_2, model, le, scaler, feature_names)
    print_result(recommendations, student_name=sample_profile_2["name"])
    print("-"*50)
    

In [154]:
if __name__ == "__main__":
    run_pipline()

--------------------------------------------------
AI - Driven Career Recommendations System
Generated dataset: 4000 samples × 83 features
Training the model ensemble
Cross-validation
CV Accuracy: 99.28% ± 0.25%
Evaluating on training data
--------------------------------------------------
Model Accuracy: 100.00
--------------------------------------------------
CLasifictation Report
                                precision    recall  f1-score   support

        Business Analyst / MBA       1.00      1.00      1.00       251
                Civil Engineer       1.00      1.00      1.00       289
                Data Scientist       1.00      1.00      1.00       264
 Doctor / Medical Professional       1.00      1.00      1.00       260
 Entrepreneur / Business Owner       1.00      1.00      1.00       282
Graphic Designer / UX Designer       1.00      1.00      1.00       257
  Journalist / Content Creator       1.00      1.00      1.00       263
   Lawyer / Legal Professional      